# 03 오믹스 다운로드 (후보 0)
코호트(1,018명) 원발 종양의 RNA(STAR counts), MAF, CNV, RPPA를 환자당 층별 1개씩 받는다.
- 이미 받은 파일은 그 경로를 그대로 쓴다: MAF·CNV·RPPA는 v4, RNA는 v3 `omics\rna_raw` (750명). 새 환자분만 받는다.
- 메틸화는 v3 QC 행렬(750명)을 쓰므로 받지 않는다. 늘리려면 `LAYERS`에 `"Meth450"` 추가.
- 파일 하나씩, 4개 병렬, md5 검증, 실패 시 재시도. 중단 후 다시 실행하면 남은 것만 받는다.
- 커널: `C:\Users\Admin\EXAONEPath\exaonepath-venv` (인터넷 필요)
- 출력: `C:\Users\Admin\EXAONEPath_data\v5\03_omics_download\{raw\, files_used.csv}`

In [1]:
# 1. 경로와 대상 파일 (환자당 층별 1개)
from pathlib import Path
import time, hashlib, requests, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

IN2  = Path(r"C:\Users\Admin\EXAONEPath_data\v5\02_gdc_inventory")
RAW4 = Path(r"C:\Users\Admin\EXAONEPath_data\v4\04_pancan_tables\raw")   # v4에서 받은 MAF·CNV·RPPA
RNA3 = Path(r"C:\Users\Admin\EXAONEPath_data\v3\omics\rna_raw")            # v3에서 받은 RNA (환자별 STAR counts)
OUT  = Path(r"C:\Users\Admin\EXAONEPath_data\v5\03_omics_download")
RAW  = OUT / "raw"
OUT.mkdir(parents=True, exist_ok=True)

LAYERS = ["RNA", "MAF", "CNV", "RPPA"]
PREF = {"ASCAT3": 0, "ASCAT2": 1, "ABSOLUTE LiftOver": 2, "AscatNGS": 3}   # CNV 워크플로 우선순위

f, co = pd.read_csv(IN2 / "gdc_files.csv"), pd.read_csv(IN2 / "cohort.csv")
f = f[(f.sample_type == "Primary Tumor") & f.patient_id.isin(co.patient_id) & f.layer.isin(LAYERS)].copy()
f["rank"] = f.workflow.map(PREF).fillna(0)
f = f.sort_values(["layer", "patient_id", "rank", "sample_id", "file_name"]).drop_duplicates(["layer", "patient_id"])

old = [RNA3 / f"{p}.tsv" if l == "RNA" else RAW4 / l / i / n
       for l, i, n, p in zip(f.layer, f.file_id, f.file_name, f.patient_id)]
f["reused"] = [p.exists() for p in old]
f["path"] = [str(o if o.exists() else RAW / l / i / n) for o, l, i, n in zip(old, f.layer, f.file_id, f.file_name)]
f.to_csv(OUT / "files_used.csv", index=False, encoding="utf-8-sig")
f.groupby("layer").agg(patients=("patient_id", "size"), reused=("reused", "sum"),
                       GB=("file_size", lambda x: round(x.sum() / 2**30, 2)))

,patients,reused,GB
layer,,,
CNV,1017,735,3.26
MAF,994,738,0.09
RNA,1018,1018,4.02
RPPA,686,517,0.01


In [2]:
# 2. 다운로드 (100개마다 진행 표시)
def get(r, tries=5):
    p = Path(r.path)
    for t in range(tries):
        try:
            x = requests.get(f"https://api.gdc.cancer.gov/data/{r.file_id}", timeout=300)
            x.raise_for_status()
            if hashlib.md5(x.content).hexdigest() != r.md5: raise IOError("md5 불일치")
            p.parent.mkdir(parents=True, exist_ok=True)
            tmp = p.with_name(p.name + ".part"); tmp.write_bytes(x.content); tmp.replace(p)
            return None
        except (requests.RequestException, IOError):
            time.sleep(15 * (t + 1))
    return r.file_id

todo = [r for r in f.itertuples() if not Path(r.path).exists()]
fail, t0 = [], time.time()
print(f"받을 파일 {len(todo)}개")
with ThreadPoolExecutor(4) as ex:
    for k, fut in enumerate(as_completed([ex.submit(get, r) for r in todo]), 1):
        if fut.result(): fail.append(fut.result())
        if k % 100 == 0 or k == len(todo):
            print(f"  {k}/{len(todo)} | {(time.time() - t0) / 60:.1f}분 | 실패 {len(fail)}", flush=True)

받을 파일 266개
  100/266 | 0.7분 | 실패 0
  200/266 | 1.3분 | 실패 0
  266/266 | 1.7분 | 실패 0


In [3]:
# 3. 확인: 층별로 모든 파일이 있는가 (실패가 있으면 2번 셀만 다시 실행)
f["ok"] = f.path.map(lambda p: Path(p).exists())
print("실패 파일:", fail or "없음")
f.groupby(["layer", "cancer"]).ok.agg(["size", "sum"]).unstack()

실패 파일: 없음


size       sum     
cancer LUAD LUSC LUAD LUSC
layer                     
CNV     516  501  516  501
MAF     507  487  507  487
RNA     517  501  517  501
RPPA    361  325  361  325